[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/125/blob/main/07_dataframes/07_2_Selecting_Rows.ipynb)

# 07.2: Selecting Rows

Notebook 07.1 selected columns. This notebook selects rows, which is how you narrow a dataset down to the part that answers your question. The main tool is the boolean mask from notebook 06.2. It works the same way here, except that it now keeps whole rows.

This notebook opens in Google Colab. If Colab warns that the notebook was not authored by Google, click **Run anyway**. Your changes are not kept when you close the tab unless you choose **File > Save a copy in Drive**, so do that if you want to keep your work.

## The data

We use the Broadway data again, where each row is one show in one week.

In [ ]:
import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_corgis_broadway.csv")
df.shape

## Filtering with a mask

A comparison on a column gives a boolean series, with one value per row. Put that mask inside the square brackets to keep the rows where it is `True`.

In [ ]:
year_2000 = df[df["Year"] == 2000]
print(year_2000.shape)
year_2000.head(3)

There are 1524 rows from the year 2000, and all 12 columns are still there. Since the result is a DataFrame, you can use any column of it. Be careful with `==`, because a single `=` means assignment and would be an error here.

As with a series, naming the mask in a separate step makes your code easier to read and to fix. The next cell gets the same rows in two steps.

In [ ]:
is_2000 = df["Year"] == 2000
print(is_2000.sum())
df.loc[is_2000, "Name"].head(2)

The mask has 1524 `True` values, which matches the 1524 rows above, and the first two shows are Amadeus and Annie Get Your Gun.

## Selecting rows and columns together: `.loc`

`.loc` from module 06 takes two things on a DataFrame, separated by a comma: which rows, and then which columns. A mask can pick the rows, and a list can pick the columns.

In [ ]:
df.loc[df["Year"] == 2000, ["Name", "Type", "Gross"]].head(3)

You can also look up one row by its index label and one column by name, which gives a single value. Using `:` for the rows means all rows.

In [ ]:
print(df.loc[0, "Name"])
print(df.loc[:, "Gross"].max())

The first line prints Tru, the show in the row labeled 0. The second prints 3201333, the largest weekly gross in the data.

Both parts of `.loc` can be slices, and the stop label is included for both, as it was with a series. `.iloc` also takes rows and then columns, using positions, and its stop position is not included. Negative numbers count from the end.

In [ ]:
print(df.loc[0:1, "Name":"Type"])
print(df.iloc[-2:, 0:2])

The first table has rows 0 and 1 and the four columns from `Name` through `Type`, and the second has the last two rows and the first two columns, `Attendance` and `Capacity`. In this course you will mostly use `.loc`, because labels say what you mean and do not depend on the order of the rows.

## Combining conditions

The rules are the same as in 06.2: use `&`, `|` and `~`, with parentheses around each condition. Which rows are musicals from the year 2000?

In [ ]:
musicals_2000 = df[(df["Type"] == "Musical") & (df["Year"] == 2000)]
print(len(musicals_2000))
print(musicals_2000["Name"].nunique())

There are 1113 rows but only 37 different musicals. Again, this is because each row is one week, so a musical that ran all year has about 52 rows. It is easy to report 1113 by accident when the question is how many musicals there were, so always ask what one row represents.

`.isin()` is the short way to match several values in one column. Here are the rows for three long-running musicals.

In [ ]:
big_three = df[df["Name"].isin(["The Lion King", "Wicked", "The Phantom Of The Opera"])]
print(len(big_three))
print(big_three["Gross"].sum())
print(df["Gross"].sum())

Those three shows account for 3119822366 of the 18654569593 dollars in the data, which is one way to answer how much of Broadway's income came from its biggest shows. We can get the share by dividing.

In [ ]:
big_three["Gross"].sum() / df["Gross"].sum()

That is about 0.167. Three of 820 shows brought in about 17 percent of all the gross in this data.

`|` keeps a row if either condition is true, and `~` flips a mask. These rows are unusual weeks, with more than 20000 people or fewer than 100.

In [ ]:
extreme = (df["Attendance"] > 20000) | (df["Attendance"] < 100)
print(extreme.sum())
print((~(df["Type"] == "Musical")).sum())

Only 14 rows are that extreme. The second line counts the 8745 rows that are not musicals, which is the plays and specials together.

## Text and masks

The `.str` methods from notebook 06.4 work on a column, and they give masks that select rows.

In [ ]:
df[df["Name"].str.contains("Phantom")]["Name"].unique()

This returns just one show, The Phantom Of The Opera. Checking with `.unique()` like this is a good way to make sure that a text search found what you meant, and did not also pick up other shows.

Combine a text search with `~` when a search picks up too much. Searching for `King` finds The Lion King and also other titles, so the next cell shows them and then leaves that one out. `.str.startswith()` works the same way.

In [ ]:
has_king = df["Name"].str.contains("King")
print(df.loc[has_king, "Name"].nunique())
print(df.loc[has_king & ~df["Name"].str.contains("Lion"), "Name"].unique()[:3])
print(df.loc[df["Name"].str.startswith("The Lion"), "Name"].unique())

Nine shows contain `King`. Leaving out names with `Lion` keeps titles such as The King And I 96 and King David. The last line finds the two shows that start with `The Lion`: The Lion King and The Lion In Winter.

## Sorting rows and finding the top N

On a DataFrame, `.sort_values()` needs to know which column to sort by. Combining it with `.head()` gives the top rows, as it did for a series.

In [ ]:
df.sort_values("Gross", ascending=False).head(3)[["Name", "Full", "Gross"]]

Those are the highest-grossing weeks in the data, and the top one is Wicked in the week of December 29, 2013, with 3201333. Each step in this chain returns a DataFrame, so you can keep adding to it. Filtering first lets you ask narrower questions, such as which musical had the largest week in 2000.

In [ ]:
musicals_2000.sort_values("Gross", ascending=False).head(1)[["Name", "Gross"]]

Aida had the largest week among musicals in 2000, at 1155646.

`.nlargest()` and `.nsmallest()` from 06.2 also work on a DataFrame, where you say which column to use. `df.nlargest(3, "Gross")` gives the same three rows as the sort above.

## Changing a filtered result

When you filter a DataFrame and then plan to change the result, such as adding a column, call `.copy()` on it. This makes clear that the result is its own table, and it prevents a warning that pandas may give otherwise. It also guarantees that `df` is not changed.

In [ ]:
plays = df[df["Type"] == "Play"].copy()
plays["Gross_Millions"] = plays["Gross"] / 1000000
plays[["Name", "Gross_Millions"]].head(2)

Tru's week is 0.134456 million dollars. To change particular values, give `.loc` a mask and a column, and assign. The next cell changes the `Type` of every special in a copy.

In [ ]:
relabeled = df.copy()
relabeled.loc[relabeled["Type"] == "Special", "Type"] = "Other"
relabeled["Type"].value_counts()

The 339 specials now say `Other`, and the counts for musicals and plays did not change.

## Putting it together: states and Grammys

Here is a worked example with one row per state, holding its estimated 2019 population and a count of Grammy winners from that state. The next cell builds the table with a method you do not need to learn, and the rest uses tools from this notebook.

In [ ]:
state_list = ['California', 'Texas', 'Florida', 'New York', 'Illinois',
              'Pennsylvania', 'Ohio', 'Georgia', 'North Carolina', 'Michigan',
              'New Jersey', 'Virginia', 'Washington', 'Arizona','Massachusetts',
              'Tennessee', 'Indiana', 'Missouri', 'Maryland', 'Wisconsin',
              'Colorado', 'Minnesota', 'South Carolina', 'Alabama', 'Louisiana',
              'Kentucky', 'Oregon', 'Oklahoma', 'Connecticut', 'Utah', 'Iowa',
              'Nevada', 'Arkansas', 'Mississippi', 'Kansas', 'New Mexico',
              'Nebraska', 'West Virginia', 'Idaho', 'Hawaii', 'New Hampshire',
              'Maine', 'Montana', 'Rhode Island', 'Delaware', 'South Dakota',
              'North Dakota', 'Alaska', 'DC', 'Vermont', 'Wyoming']

population_list = [39512223, 28995881, 21477737, 19453561, 12671821, 12801989,
                   11689100, 10617423, 10488084, 9986857, 8882190, 8535519,
                   7614893, 7278717, 6949503, 6833174, 6732219, 6137428,
                   6045680, 5822434, 5758736, 5639632, 5148714, 4903185,
                   4648794, 4467673, 4217737, 3956971, 3565287, 3205958,
                   3155070, 3080156, 3017825, 2976149, 2913314, 2096829,
                   1934408, 1792147, 1787065, 1415872, 1359711, 1344212,
                   1068778, 1059361, 973764, 884659, 762062, 731545, 705749,
                   623989, 578759]

# the Grammy counts are in alphabetical order by state
grammy_list = [3,0,0,3,27,1,1,None,0,2,10,3,0,15,3,1,1,1,1,2,3,4,4,3,2,5,0,1,
               0,0,15,0,49,3,0,8,2,1,7,1,1,1,7,12,0,0,3,3,0,1,0] 


states = pd.DataFrame({"state": state_list, "population": population_list})
states = states.sort_values("state")
states["grammies"] = grammy_list
states = states.set_index("state")
states.head(3)

Alabama has 3 winners, and Alaska and Arizona have none. The counts show as 3.0 and 0.0 because D.C.'s count is missing, and a numeric column with a missing value is stored with decimal places. Which states have the most?

In [ ]:
states.sort_values("grammies", ascending=False).head(3)

New York leads with 49, followed by California with 27 and New Jersey with 15. Now the most populous states with no winners, using a mask and then a sort.

In [ ]:
states[states["grammies"] == 0].sort_values("population", ascending=False).head(3)

Arizona is the most populous state with no winners in this list, followed by Utah and Nevada. A raw count favors large states, so a fairer comparison is winners per million people. This is math between two columns, which lines up by label as you saw in 06.2.

In [ ]:
per_million = states["grammies"] / states["population"] * 1000000
per_million.sort_values(ascending=False).head(3)

New York still leads at about 2.52 per million, but Hawaii moves up to second at about 2.12, even though it has far fewer winners than California.

## Practice problems

These problems are not collected or graded. They are here so that you can find out whether you can write this code yourself before the reading check and the pair activity. Write your answer in the cell under each problem, run it, and then run the check cell below it, which tells you whether your answer is right. The check cells are collapsed because they contain the answers, so try the problem before you open one.

Run the next two cells first. The first one creates fresh data for the problems, and the second sets up the checks.

In [ ]:
import pandas as pd

albums = pd.read_csv("https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_albums_sales_wikipedia.csv")
albums_start = albums.copy()

In [ ]:
#@title Run this cell to set up the checks
import numpy as np
import pandas as pd


def same(got, want):
    """True when got matches want: same labels and values for a series, same value for a number."""
    try:
        if isinstance(want, pd.DataFrame):
            if not isinstance(got, pd.DataFrame) or list(got.columns) != list(want.columns) or len(got) != len(want):
                return False
            if not pd.api.types.is_integer_dtype(want.index) and list(got.index) != list(want.index):
                return False
            got, want = got.reset_index(drop=True), want.reset_index(drop=True)
            for column in want.columns:
                if pd.api.types.is_numeric_dtype(want[column]):
                    if not np.allclose(pd.to_numeric(got[column], errors="coerce").astype(float), want[column].astype(float), equal_nan=True):
                        return False
                elif got[column].fillna("").astype(str).to_list() != want[column].fillna("").astype(str).to_list():
                    return False
            return True
        if isinstance(want, pd.Series):
            if not isinstance(got, pd.Series) or list(got.index) != list(want.index):
                return False
            if pd.api.types.is_numeric_dtype(want):
                return bool(np.allclose(got.astype(float), want.astype(float), equal_nan=True))
            return list(got) == list(want)
        if isinstance(want, (list, np.ndarray)):
            return list(got) == list(want)
        if isinstance(want, str):
            return got == want
        if isinstance(got, (pd.Series, pd.DataFrame, list, np.ndarray)):
            return False
        return bool(np.isclose(float(got), float(want)))
    except (TypeError, ValueError):
        return False


def check(number, name, want, hint, unchanged=None):
    """Print feedback for one problem. unchanged is (variable name, its starting value)."""
    if name not in globals():
        print(f"Problem {number}: not yet. No variable named {name} exists. Did you run your answer cell?")
    elif not same(globals()[name], want):
        print(f"Problem {number}: not yet. {hint}")
    elif unchanged is not None and not globals()[unchanged[0]].equals(unchanged[1]):
        print(f"Problem {number}: {name} is right, but {unchanged[0]} changed. Work on a copy, then run the setup cell again.")
    else:
        print(f"Problem {number}: correct.")

### Problem 1
Store the `year` from the row labeled 0 in a variable named `first_year`, using `.loc`.

In [ ]:
# Problem 1

In [ ]:
#@title Check Problem 1
check(1, "first_year", albums_start.loc[0, "year"], "Give .loc the row label and then the column name.")

### Problem 2
Create a DataFrame named `since_1990` with only the rows where `year` is 1990 or later. An album from exactly 1990 counts.

In [ ]:
# Problem 2

In [ ]:
#@title Check Problem 2
check(2, "since_1990", albums_start[albums_start["year"] >= 1990], "Use a mask with >=.")

### Problem 3
Create a DataFrame named `big_jackson` with only the rows where `artist` is `Michael Jackson` and `cert_sales` is above 20.

In [ ]:
# Problem 3

In [ ]:
#@title Check Problem 3
check(3, "big_jackson", albums_start[(albums_start["artist"] == "Michael Jackson") & (albums_start["cert_sales"] > 20)], "Combine two conditions with &, and put parentheses around each.")

### Problem 4
Use `.isin()` to create a DataFrame named `two_acts` with only the rows where `artist` is `Eagles` or `The Beatles`.

In [ ]:
# Problem 4

In [ ]:
#@title Check Problem 4
check(4, "two_acts", albums_start[albums_start["artist"].isin(["Eagles", "The Beatles"])], "Pass .isin() a list of the two names.")

### Problem 5
Create a DataFrame named `not_beatles` with every row except those where `artist` is `The Beatles`, using `~`.

In [ ]:
# Problem 5

In [ ]:
#@title Check Problem 5
check(5, "not_beatles", albums_start[~(albums_start["artist"] == "The Beatles")], "Put ~ in front of a mask in parentheses.")

### Problem 6
Create a DataFrame named `top_three` with the three rows with the highest `claim_sales`, ordered from highest to lowest.

In [ ]:
# Problem 6

In [ ]:
#@title Check Problem 6
check(6, "top_three", albums_start.sort_values("claim_sales", ascending=False).head(3), "Sort by the column from high to low and take the first three.")

### Problem 7
Create a DataFrame named `pop_albums` with only the rows where `genre` contains the text `Pop`, with a capital P.

In [ ]:
# Problem 7

In [ ]:
#@title Check Problem 7
check(7, "pop_albums", albums_start[albums_start["genre"].str.contains("Pop")], "Use .str.contains() on the genre column as a mask.")

### Problem 8
Create a DataFrame named `corner` with the first five rows and the first two columns, using `.iloc`.

In [ ]:
# Problem 8

In [ ]:
#@title Check Problem 8
check(8, "corner", albums_start.iloc[:5, :2], "Give .iloc a slice of rows and then a slice of columns.")

### Problem 9
Create a copy of `albums` named `fixed`, and in the copy change `claim_sales` to 55 in the row where `album` is `Back in Black`. `albums` should not change.

In [ ]:
# Problem 9

In [ ]:
#@title Check Problem 9
fixed_want = albums_start.copy()
fixed_want.loc[fixed_want["album"] == "Back in Black", "claim_sales"] = 55
check(9, "fixed", fixed_want, "Use .copy(), then give .loc a mask and the column name.", ("albums", albums_start))

## What's next

You can now pull out any rows and columns you want. Before you trust an answer, though, you need to know whether the data behind it is sound, and you saw a warning sign in 07.1 with a `Capacity` of 892. Notebook 07.3 shows how to spot problems like this one and what to do about them.